# « Développez une preuve de concept »

Projet n&#8239;$^\text{o}$ 7 du [cursus Machine Learning Engineer][2] d'OpenClassrooms

Auteur : [Kiril ISAKOV][1]

Mentor : Nicolas TISSERAND

Projet démarré le 18/05/2026

[1]: https://github.com/kirisakow/
[2]: https://openclassrooms.com/fr/paths/794-machine-learning-engineer

# Notebook d’entraînement du modèle SoTA `YOLO26` à 120 classes

Le dataset : http://vision.stanford.edu/aditya86/ImageNetDogs/

## Imports et constantes

In [ ]:
import os
# Déclarer le backend avant d'importer keras (sinon par défaut c'est 'tensorflow'):
os.environ["KERAS_BACKEND"] = "torch"
# Memory optimization: Enable expandable segments to reduce fragmentation:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# Limiter la verbosité des logs de tensorflow:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from functions_model_building import (
    plot_confusion_matrix,
    print_classification_report,
    temp_dir_with_symlinks,
    WORDNET_ID_REGEX_PTRN
)
from collections import Counter
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm.notebook import tqdm
from ultralytics import YOLO
import datetime as dt
import itertools as it
import logging
import numpy as np
import pandas as pd
import pytz
import re
import time
import torch
import warnings

# logging.getLogger('tensorflow').setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

LOGGER_FORMAT = '%(asctime)s [%(levelname)s] %(message)s'
logging.Formatter.converter = lambda *_: dt.datetime.now(pytz.timezone('Europe/Paris')).timetuple()
logging.basicConfig(level=logging.INFO, format=LOGGER_FORMAT, force=True)
logr = logging.getLogger(__name__)
logr.setLevel(logging.DEBUG)

DEFAULT_BATCH_SIZE = 8
DEFAULT_TOP_K_CATEG_ACCUR = 5
DEFAULT_TRGT_IMG_SIZE = (224, 224)
DO_NOT_AUTOSAVE = -1
LEARNING_RATE_LOW = 1e-5
LEARNING_RATE_REGULAR = 1e-2
N_EPOCHS = 20
PRETR_MDL_LBL = 'yolo26n-cls'

## Split train - val - test

In [2]:
input_img_paths = tuple(
    # [Path('images/n02090379-redbone'), Path('images/n02085936-Maltese_dog'), Path('images/n02088094-Afghan_hound')]
    Path('images/').glob('*/')
)
logr.info(f"Nombre de classes retenues pour l'entraînement : {len(input_img_paths)}")
input_img_paths = tuple(path.glob('*.jpg') for path in input_img_paths)
input_img_paths = tuple(it.chain.from_iterable(input_img_paths))
logr.info(f"Résultat : un tuple de {len(input_img_paths)} images pour les 3 classes de races.")
breed_labels = tuple(str(path).split('/')[1] for path in input_img_paths)
breed_labels = tuple(WORDNET_ID_REGEX_PTRN.split(label_with_id)[1] for label_with_id in breed_labels)
n = 1000
logr.info(f"Un échantillon de la distribution des classes (seuls les {n} premiers chemins sont comptés) :")
display(dict(Counter(breed_labels[:n])))
train_frac = 0.8
input_paths_train, input_paths_val_test, labels_train, labels_val_test = train_test_split(
    input_img_paths, breed_labels, train_size=train_frac,
    random_state=42, shuffle=True, stratify=breed_labels,
)
val_test_frac = 0.5
input_paths_val, input_paths_test, labels_val, labels_test = train_test_split(
    input_paths_val_test, labels_val_test, train_size=val_test_frac,
    random_state=42, shuffle=True, stratify=labels_val_test
)
del input_paths_val_test, labels_val_test, n
logr.info(f"Taile de l'échantillon de train ({train_frac}): {len(input_paths_train)}")
logr.info(f"Taile de l'échantillon de val ({round(val_test_frac * (1 - train_frac), 1)}): {len(input_paths_val)}")
logr.info(f"Taile de l'échantillon de test ({round(val_test_frac * (1 - train_frac), 1)}): {len(input_paths_test)}")

2026-06-24 12:18:14,155 [INFO] Nombre de classes retenues pour l'entraînement : 120
2026-06-24 12:18:14,775 [INFO] Résultat : un tuple de 20580 images pour les 3 classes de races.
2026-06-24 12:18:14,792 [INFO] Un échantillon de la distribution des classes (seuls les 1000 premiers chemins sont comptés) :


{'briard': 152,
 'vizsla': 154,
 'kuvasz': 150,
 'papillon': 196,
 'African_hunting_dog': 169,
 'Great_Pyrenees': 179}

2026-06-24 12:18:14,827 [INFO] Taile de l'échantillon de train (0.8): 16464
2026-06-24 12:18:14,828 [INFO] Taile de l'échantillon de val (0.1): 2058
2026-06-24 12:18:14,829 [INFO] Taile de l'échantillon de test (0.1): 2058


## Entraînement

### Déclarer les paramètres

In [3]:
N_CLASSES = len(set(breed_labels))
CLASS_NAMES_SORTED = sorted(set(breed_labels))
FT = 10
TOP_K_CATEG_ACCUR = min(DEFAULT_TOP_K_CATEG_ACCUR, N_CLASSES - 1)
TEMP_DIR_WITH_SYMLINKS = temp_dir_with_symlinks(input_paths_train, input_paths_val, input_paths_test)

### Training: Stage 1: freeze backbone, train head and neck

In [4]:
experiment_name = (
    f'CNN__from={PRETR_MDL_LBL}'
    f'__n_cls={N_CLASSES}'
    f'__n_eps={N_EPOCHS}'
    f'__LR={LEARNING_RATE_REGULAR}'
    f'__FT={FT}'
    f'__stage1'
)
PATH_TO_MDL_STG1 = f'models/{experiment_name}.pt'
if os.path.exists(PATH_TO_MDL_STG1):
    logr.info(f"Loading saved model {PATH_TO_MDL_STG1!r}")
    model = YOLO(PATH_TO_MDL_STG1)
else:
    logr.info(f"Training {experiment_name}")
    logr.info("Stage 1: freeze backbone, train head and neck")
    model = YOLO(f'{PRETR_MDL_LBL}.pt')
    results = model.train(
        batch=DEFAULT_BATCH_SIZE,
        data=TEMP_DIR_WITH_SYMLINKS,
        device=0 if torch.cuda.is_available() else 'cpu',
        epochs=N_EPOCHS,
        exist_ok=True,
        freeze=FT,
        imgsz=DEFAULT_TRGT_IMG_SIZE[0],
        lr0=LEARNING_RATE_REGULAR,
        name=experiment_name,
        save_dir=f"log/{experiment_name}",
        save_period=DO_NOT_AUTOSAVE,
        verbose=True,
    )
    logr.info("Training Stage 1 complete.")
    logr.info(f"Saving model to {PATH_TO_MDL_STG1!r}")
    model.save(PATH_TO_MDL_STG1)

2026-06-24 12:23:33,528 [INFO] Training CNN__from=yolo26n-cls__n_cls=120__n_eps=20__LR=0.01__FT=10__stage1
2026-06-24 12:23:33,529 [INFO] Stage 1: freeze backbone, train head and neck


Ultralytics 8.4.75 🚀 Python-3.12.3 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 2070 with Max-Q Design, 7778MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/tmp/ultralytics_tempdir_mpm51zrm, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=CNN__from=yolo26n-cls__n_cls=120__n_eps=20__LR=0.01__FT=10__stage1, nb

2026-06-24 12:40:46,525 [INFO] Training Stage 1 complete.
2026-06-24 12:40:46,526 [INFO] Saving model to 'models/CNN__from=yolo26n-cls__n_cls=120__n_eps=20__LR=0.01__FT=10__stage1.pt'


### Training: Stage 2: unfreeze all, fine-tune with lower learning rate

In [5]:
torch.cuda.empty_cache()

In [6]:
experiment_name = (
    f'CNN__from={PRETR_MDL_LBL}'
    f'__n_cls={N_CLASSES}'
    f'__n_eps={N_EPOCHS}'
    f'__LR={LEARNING_RATE_LOW}'
    f'__FT={FT}'
    f'__stage2'
)
PATH_TO_MDL_STG2 = f'models/{experiment_name}.pt'
if os.path.exists(PATH_TO_MDL_STG2):
    logr.info(f"Loading saved model {PATH_TO_MDL_STG2!r}")
    model = YOLO(PATH_TO_MDL_STG2)
else:
    logr.info(f"Training {experiment_name}")
    logr.info("Stage 2: unfreeze whole backbone, fine-tune with lower LR")
    model = YOLO(PATH_TO_MDL_STG1)
    results = model.train(
        batch=DEFAULT_BATCH_SIZE,
        data=TEMP_DIR_WITH_SYMLINKS,
        device=0 if torch.cuda.is_available() else 'cpu',
        epochs=N_EPOCHS,
        exist_ok=True,
        imgsz=DEFAULT_TRGT_IMG_SIZE[0],
        lr0=LEARNING_RATE_LOW,
        name=experiment_name,
        save_dir=f"log/{experiment_name}",
        save_period=DO_NOT_AUTOSAVE,
        verbose=True,
    )
    logr.info("Training Stage 2 complete.")
    logr.info(f"Saving model to {PATH_TO_MDL_STG2!r}")
    model.save(PATH_TO_MDL_STG2)

2026-06-24 12:41:57,148 [INFO] Training CNN__from=yolo26n-cls__n_cls=120__n_eps=20__LR=1e-05__FT=10__stage2
2026-06-24 12:41:57,148 [INFO] Stage 2: unfreeze whole backbone, fine-tune with lower LR


Ultralytics 8.4.75 🚀 Python-3.12.3 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 2070 with Max-Q Design, 7778MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/tmp/ultralytics_tempdir_mpm51zrm, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1e-05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=models/CNN__from=yolo26n-cls__n_cls=120__n_eps=20__LR=0.01__FT=10__stage1.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=CNN__

2026-06-24 13:10:20,544 [INFO] Training Stage 2 complete.
2026-06-24 13:10:20,544 [INFO] Saving model to 'models/CNN__from=yolo26n-cls__n_cls=120__n_eps=20__LR=1e-05__FT=10__stage2.pt'


## Évaluation sur les données de test

In [7]:
torch.cuda.empty_cache()

### Metrics

In [8]:
logr.info("Evaluating on test set...")
metrics = model.val(
    batch=DEFAULT_BATCH_SIZE,
    data=TEMP_DIR_WITH_SYMLINKS,
    device=0 if torch.cuda.is_available() else 'cpu',
    exist_ok=True,
    save_dir=f"log/{experiment_name}",
    save_period=DO_NOT_AUTOSAVE,
    split='test',
    verbose=True,
)
logr.info(f"Metrics :\n\n{metrics}")

2026-06-22 22:06:19,560 [INFO] Evaluating on test set...


Ultralytics 8.4.75 🚀 Python-3.12.3 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 2070 with Max-Q Design, 7778MiB)
YOLO26n-cls summary (fused): 47 layers, 1,529,867 parameters, 0 gradients, 3.2 GFLOPs
train: /tmp/ultralytics_tempdir_p3iggitm/train... found 511 images in 3 classes ✅ 
val: /tmp/ultralytics_tempdir_p3iggitm/val... found 64 images in 3 classes ✅ 
test: /tmp/ultralytics_tempdir_p3iggitm/test... found 64 images in 3 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1151.0±408.1 MB/s, size: 43.6 KB)
test: Scanning /tmp/ultralytics_tempdir_p3iggitm/test... 64 images, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 4.1Kit/s 0.0s
test: New cache created: /tmp/ultralytics_tempdir_p3iggitm/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 8/8 27.4it/s 0.3s.7s
                   all      0.969          1
Speed: 0.3ms preprocess, 3.6ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /home/mira/kiril_dev/oc-dogs-cv-dl-poc/log/CNN__from=yolo2

2026-06-22 22:06:21,333 [INFO] Metrics :

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78b69c0cfd40>
curves: []
curves_results: []
fitness: 0.984375
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.96875, 'metrics/accuracy_top5': 1.0, 'fitness': 0.984375}
save_dir: PosixPath('/home/mira/kiril_dev/oc-dogs-cv-dl-poc/log/CNN__from=yolo26n-cls__n_cls=3__n_eps=20__LR=1e-05__FT=10__stage2')
speed: {'preprocess': 0.33011074998512413, 'inference': 3.586263984402649, 'loss': 0.003101203105870809, 'postprocess': 0.003360718778822047}
top1: 0.96875
top5: 1.0


### Classification Report & Confusion Matrix

In [ ]:
y_true = []
y_pred_probs = []
y_pred_topk = []
for path, label in tqdm(zip(input_paths_test, labels_test), total=len(input_paths_test), leave=False):
    results = model(path, verbose=False)
    probs = results[0].probs.data.cpu().numpy()
    y_pred_probs.append(probs)
    true_idx = CLASS_NAMES_SORTED.index(label)
    y_true.append(true_idx)
    top_k_idx = np.argpartition(probs, -TOP_K_CATEG_ACCUR)[-TOP_K_CATEG_ACCUR:]
    y_pred_topk.append(true_idx in top_k_idx)
y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred_probs)
print("\nClassification Report:\n")
print_classification_report(y_true_arr, y_pred_arr, class_labels=CLASS_NAMES_SORTED)
print("\nConfusion Matrix:\n")
plot_confusion_matrix(y_pred_arr, y_true_arr, CLASS_NAMES_SORTED, title=experiment_name)
top_k_accuracy = sum(y_pred_topk) / len(y_pred_topk)
print(f"\nTop-k Accuracy: {top_k_accuracy:.4f}\n")

  0%|          | 0/2058 [00:00<?, ?it/s]


Classification Report:

                                precision    recall  f1-score   support

                  Afghan_hound       0.91      0.83      0.87        24
           African_hunting_dog       1.00      1.00      1.00        17
                      Airedale       0.82      0.90      0.86        20
American_Staffordshire_terrier       0.69      0.53      0.60        17
                   Appenzeller       0.70      0.47      0.56        15
            Australian_terrier       0.65      0.65      0.65        20
            Bedlington_terrier       0.90      1.00      0.95        18
          Bernese_mountain_dog       0.88      0.95      0.91        22
              Blenheim_spaniel       0.94      0.84      0.89        19
                 Border_collie       0.65      0.73      0.69        15
                Border_terrier       0.68      1.00      0.81        17
                   Boston_bull       0.76      0.89      0.82        18
          Bouvier_des_Flandres       0

<Figure size 600x400 with 2 Axes>


Top-k Accuracy: 0.9660



## Prédiction en mode *blind test*

Effectuer l'inférence :

In [13]:
DEFAULT_BLIND_TEST_SAMPLE_SIZE = 100
blind_test_sample = np.random.choice(
    input_paths_test,
    size=min(DEFAULT_BLIND_TEST_SAMPLE_SIZE, len(input_paths_test)),
    replace=False
)
start_time = time.perf_counter()
predicted_class_idx = []
for path in blind_test_sample:
    results = model(path, verbose=False)
    pred_idx = results[0].probs.top5[0]
    predicted_class_idx.append(pred_idx)
end_time = time.perf_counter()
print(f"\nInference time on a sample of size {len(blind_test_sample)}: {end_time - start_time:.2f}s")


Inference time on a sample of size 100: 0.81s


Afficher les résultats sous une forme conviviale :

In [14]:
df = pd.DataFrame({
    'path': blind_test_sample,
    'predicted_breed': [CLASS_NAMES_SORTED[pred_idx] for pred_idx in predicted_class_idx]
}, dtype=str)
df['is_correct'] = df.apply(
    lambda row: '✅' if row['predicted_breed'] in row['path'] else '❗️',
    axis=1
)
default_max_rows = pd.options.display.max_rows
pd.options.display.max_rows = DEFAULT_BLIND_TEST_SAMPLE_SIZE
display(df)
pd.options.display.max_rows = default_max_rows

,path,predicted_breed,is_correct
0,images/n02107683-Bernese_mountain_dog/n0210768...,Bernese_mountain_dog,✅
1,images/n02099849-Chesapeake_Bay_retriever/n020...,Chesapeake_Bay_retriever,✅
2,images/n02094114-Norfolk_terrier/n02094114_386...,Norfolk_terrier,✅
3,images/n02107683-Bernese_mountain_dog/n0210768...,Bernese_mountain_dog,✅
4,images/n02099712-Labrador_retriever/n02099712_...,Labrador_retriever,✅
5,images/n02102177-Welsh_springer_spaniel/n02102...,Appenzeller,❗️
6,images/n02108915-French_bulldog/n02108915_1343...,French_bulldog,✅
7,images/n02100735-English_setter/n02100735_4813...,English_setter,✅
8,images/n02099267-flat-coated_retriever/n020992...,Gordon_setter,❗️
9,images/n02099712-Labrador_retriever/n02099712_...,French_bulldog,❗️
